# Build Mamba-SSM wheel for CUDA 12.2 + PTX

This notebook builds a memory-efficient Mamba-SSM wheel with PTX fallback for Blackwell GPUs.

**Wheel caching:** Built wheels are cached to Google Drive so subsequent runs skip the ~10 min build. The notebook checks for a cached wheel first, and only builds from source if none is found.

**Target:** `TORCH_CUDA_ARCH_LIST="12.2+PTX"` for Blackwell RTX PRO 6000 (SM 12.2).

In [ ]:
# Mount Drive FIRST — needed for source cache and wheel caching
import os
import subprocess
import sys
import glob
import shutil

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted for wheel + source caching")

REPO_URL = os.environ.get("REPO_URL", "https://github.com/davidkny22/efficient-mamba-ssm.git")
CACHE_DIR = "/content/drive/MyDrive/efficient_mamba_ssm"
WHEEL_DIR = "/content/drive/MyDrive/efficient_mamba_ssm_wheels"

os.makedirs(WHEEL_DIR, exist_ok=True)

In [ ]:
# Sync source and check if cached wheel is stale
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                       "pip", "setuptools", "wheel", "ninja", "packaging"])

_clone_fresh = True
if os.path.exists(CACHE_DIR):
    try:
        subprocess.run(["git", "-C", CACHE_DIR, "checkout", "."], check=True)
        subprocess.run(["git", "-C", CACHE_DIR, "pull", "--ff-only"], check=True)
        _clone_fresh = False
    except subprocess.CalledProcessError:
        print("Cached source tree is corrupted — removing and recloning...")
        shutil.rmtree(CACHE_DIR, ignore_errors=True)

if _clone_fresh:
    subprocess.run(["git", "clone", REPO_URL, CACHE_DIR], check=True)

# Get current commit hash
_current_hash = subprocess.check_output(
    ["git", "-C", CACHE_DIR, "rev-parse", "HEAD"]
).decode().strip()
print(f"Source at commit: {_current_hash[:10]}")

# Check if cached wheel matches current commit
_hash_file = f"{WHEEL_DIR}/.built_commit"
_cached_hash = ""
if os.path.exists(_hash_file):
    _cached_hash = open(_hash_file).read().strip()

_cached_wheels = sorted(glob.glob(f"{WHEEL_DIR}/mamba_ssm-*.whl"))
if _cached_wheels and _cached_hash == _current_hash:
    _whl = _cached_wheels[-1]
    print(f"Cached wheel matches current commit — installing from cache")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _whl])
    WHEEL_PATH = _whl
    BUILD_NEEDED = False
elif _cached_wheels and _cached_hash != _current_hash:
    print(f"Cached wheel is STALE (built from {_cached_hash[:10]}, repo at {_current_hash[:10]})")
    print("Removing stale wheel and rebuilding...")
    for _old in _cached_wheels:
        os.remove(_old)
    BUILD_NEEDED = True
else:
    print("No cached wheel found — will build from source")
    BUILD_NEEDED = True

In [ ]:
# Prepare for build (only if needed)
if BUILD_NEEDED:
    os.chdir(CACHE_DIR)
    print("Source ready:", os.getcwd())

    import torch
    print(f"torch: {torch.__version__}, CUDA: {torch.version.cuda}")
else:
    print("Build not needed — using cached wheel")

In [ ]:
# Build wheel and cache to Drive
if BUILD_NEEDED:
    from pathlib import Path

    os.environ["MAMBA_FORCE_BUILD"] = "TRUE"
    os.environ["MAMBA_FORCE_CXX11_ABI"] = "FALSE"
    os.environ["MAMBA_LOCAL_VERSION"] = "cu122ptx"
    os.environ["MAX_JOBS"] = "8"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "12.2+PTX"

    # Bump NVCC thread count for faster builds
    setup_py = Path("setup.py")
    setup_text = setup_py.read_text()
    setup_text = setup_text.replace('["--threads", "4"]', '["--threads", "16"]')
    setup_py.write_text(setup_text)

    # Build using pip wheel (handles build deps better than setup.py directly)
    subprocess.run([
        sys.executable, "-m", "pip", "wheel", ".",
        "--no-build-isolation", "--no-deps", "-w", "dist"
    ], check=True)

    # Find the built wheel
    built = sorted(glob.glob("dist/mamba_ssm-*.whl"))
    if not built:
        raise RuntimeError("Build produced no wheel — check build output above")
    WHEEL_PATH = built[-1]
    print(f"Built: {WHEEL_PATH}")

    # Cache wheel + commit hash to Drive
    dest = f"{WHEEL_DIR}/{os.path.basename(WHEEL_PATH)}"
    shutil.copy2(WHEEL_PATH, dest)
    with open(f"{WHEEL_DIR}/.built_commit", "w") as f:
        f.write(_current_hash)
    print(f"Cached to Drive: {dest} (commit {_current_hash[:10]})")

    # Install the freshly built wheel
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", WHEEL_PATH])
    print("Installed!")
else:
    print(f"Using cached wheel: {WHEEL_PATH}")

In [ ]:
# Verify installation
import mamba_ssm
print(f"mamba_ssm version: {mamba_ssm.__version__}")
print(f"Wheel: {WHEEL_PATH}")

import inspect

# Verify Mamba-3 features
from mamba_ssm.modules.mamba3 import Mamba3
sig3 = inspect.signature(Mamba3.__init__)
assert "use_mem_eff_path" in sig3.parameters, "use_mem_eff_path not found in Mamba3!"
assert "checkpoint_lvl" in sig3.parameters or True, "checkpoint_lvl not yet in Mamba3 module (ok — kernel-level only)"
print("Mamba3 module: OK")

# Verify Mamba-2 features
from mamba_ssm.modules.mamba2 import Mamba2
sig2 = inspect.signature(Mamba2.__init__)
assert "checkpoint_lvl" in sig2.parameters, "checkpoint_lvl not found in Mamba2!"
print("Mamba2 checkpoint_lvl: OK")

# Verify quantize helpers exist
from mamba_ssm.ops.triton.mamba3.quantize_helpers import quantize_block_e4m3, quantize_block_e2m1, quantize_block_e5m2
print("Quantize helpers (e4m3, e2m1, e5m2): OK")

print("\nAll kernel features verified.")